# Evaluation results
Load saved episodes from the evaluation runs


In [12]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt


Load every `episodes.csv` in `results/evaluation/`.


In [13]:
results_dir = Path("../../results/evaluation")
if not results_dir.exists():
    results_dir = Path("results/evaluation")
result_files = sorted(results_dir.glob("*/episodes.csv"))
assert result_files, f"No evaluation CSVs found in {results_dir}"
results = pd.concat([pd.read_csv(path) for path in result_files], ignore_index=True)
results.head()


,claim,environment,dataset,checkpoint,K,training_seed,target_return,episode_seed,episode_return,episode_length
0,2,ant,medium-replay-v2,k1_seed42,1,42,2700,100,3383.733808,1000
1,2,ant,medium-replay-v2,k1_seed42,1,42,2700,101,3382.300228,1000
2,2,ant,medium-replay-v2,k1_seed42,1,42,2700,102,3637.169453,1000
3,2,ant,medium-replay-v2,k1_seed42,1,42,2700,103,3575.340455,1000
4,2,ant,medium-replay-v2,k1_seed42,1,42,2700,104,3080.631016,1000


Average results for environtment + dataset + training seed

In [14]:
groups = ["claim", "environment", "dataset", "K", "target_return"]

per_model = results.groupby(groups + ["checkpoint", "training_seed"]).agg(
    episodes=("episode_return", "count"),
    mean_return=("episode_return", "mean"),
    episode_std=("episode_return", "std"),
    mean_length=("episode_length", "mean"),
).reset_index()

per_model


,claim,environment,dataset,K,target_return,checkpoint,training_seed,episodes,mean_return,episode_std,mean_length
0,2,ant,medium-replay-v2,1,2700,k1_seed42,42,10,3438.435403,176.058692,990.7
1,2,ant,medium-replay-v2,1,2700,k1_seed43,43,10,3370.644614,262.586298,999.8
2,2,ant,medium-replay-v2,1,2700,k1_seed44,44,10,3523.658821,156.301561,1000.0


Then average the model means across training seeds. `seed_std` is undefined (`NaN`) with one model; it is not episode variability.


In [15]:
summary = per_model.groupby(groups).agg(
    models=("mean_return", "count"),
    mean_return=("mean_return", "mean"),
    seed_std=("mean_return", "std"),
).reset_index()

# D4RL random/expert reference scores; v2 uses the same values as v0.
random_scores = {"ant": -325.6, "halfcheetah": -280.178953, "hopper": -20.272305}
expert_scores = {"ant": 3879.7, "halfcheetah": 12135.0, "hopper": 3234.3}
random_score = summary["environment"].map(random_scores)
expert_score = summary["environment"].map(expert_scores)
scale = 100 / (expert_score - random_score)
summary["normalized_target"] = (summary["target_return"] - random_score) * scale
summary["normalized_return"] = (summary["mean_return"] - random_score) * scale
summary["normalized_seed_std"] = summary["seed_std"] * scale
summary


,claim,environment,dataset,K,target_return,models,mean_return,seed_std,normalized_target,normalized_return,normalized_seed_std
0,2,ant,medium-replay-v2,1,2700,3,3444.246279,76.67243,71.947305,89.645121,1.823233


## Claim 1  Return conditioning
Preparing data for plotting the results of return conditioning. Assesing diferent targets.
The different targets where choosen because of the target analysis done on the different notebook in which for every environtment, dataset we checked which target values would be expected. We use different target values that fall in the range of returns seen on the training data, there is ane extra target return to see if the model is capable of achieving higher returns that what was seen in the training trajectories.


In [16]:
claim1 = summary[summary["claim"] == 1].copy()
claim1["absolute_target_error"] = (claim1["mean_return"] - claim1["target_return"]).abs()
if claim1.empty:
    print("No Claim 1 results yet.")
claim1


No Claim 1 results yet.


,claim,environment,dataset,K,target_return,models,mean_return,seed_std,normalized_target,normalized_return,normalized_seed_std,absolute_target_error


In [ ]:
for (environment, dataset, k), data in claim1.groupby(["environment", "dataset", "K"]):
    data = data.sort_values("normalized_target")
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.plot(data["normalized_target"], data["normalized_return"], "o-", label="Achieved return")
    with_std = data.dropna(subset=["normalized_seed_std"])
    if not with_std.empty:
        ax.errorbar(with_std["normalized_target"], with_std["normalized_return"],
                    yerr=with_std["normalized_seed_std"], fmt="none", capsize=3)
    limits = [min(data["normalized_target"].min(), data["normalized_return"].min()),
              max(data["normalized_target"].max(), data["normalized_return"].max())]
    ax.plot(limits, limits, "--", color="gray", label="Achieved = requested")
    ax.set(title=f"{environment} / {dataset} / K={k}",
           xlabel="Requested return (normalized)",
           ylabel="Mean achieved return (normalized)")
    ax.legend()
    plt.tight_layout()
    plt.show()


A small error means the average achieved return is close to the target. The table below averages this error equally across tested targets; it is not the average absolute error of individual episodes.


In [18]:
claim1.groupby(["environment", "dataset", "K"]).agg(
    targets=("target_return", "count"),
    mean_absolute_target_error=("absolute_target_error", "mean"),
)


,,,targets,mean_absolute_target_error
environment,dataset,K,,


## Claim 2  Context length
Compare K=1 and K=20 at the same target using the table below.


In [19]:
claim2 = summary[summary["claim"] == 2].copy()
claim2


,claim,environment,dataset,K,target_return,models,mean_return,seed_std,normalized_target,normalized_return,normalized_seed_std
0,2,ant,medium-replay-v2,1,2700,3,3444.246279,76.67243,71.947305,89.645121,1.823233


Pair the two K values by training seed before computing their difference. Positive values favour K=20. 


In [20]:
paired = per_model[per_model["claim"] == 2].pivot(
    index=["environment", "dataset", "target_return", "training_seed"],
    columns="K", values="mean_return",
).reindex(columns=[1, 20])
paired["difference_K20_minus_K1"] = paired[20] - paired[1]
paired


K                                                                   1  20  \
environment dataset          target_return training_seed                    
ant         medium-replay-v2 2700          42             3438.435403 NaN   
                                           43             3370.644614 NaN   
                                           44             3523.658821 NaN   

K                                                         difference_K20_minus_K1  
environment dataset          target_return training_seed                           
ant         medium-replay-v2 2700          42                                 NaN  
                                           43                                 NaN  
                                           44                                 NaN

In [21]:
context_comparison = paired.groupby(level=["environment", "dataset", "target_return"]).agg(
    paired_seeds=("difference_K20_minus_K1", "count"),
    mean_difference=("difference_K20_minus_K1", "mean"),
    seed_std=("difference_K20_minus_K1", "std"),
)
context_comparison


,,,paired_seeds,mean_difference,seed_std
environment,dataset,target_return,,,
ant,medium-replay-v2,2700,0,NaN,NaN
